# Diagnose llcRental Accounting Books 

 

In [6]:
import os
from pathlib import Path
import json
import pandas as pd
from IPython.display import display, Markdown

In [2]:
### -- ExpRev Global Change

# import json



In [60]:
# llcExpRev Data
import pandas as pd
from IPython.display import display, Markdown

fnDict = dict(asset = Path.home()/ 'GDrive/Family/Assets/LLC-WBGroup/books/Accts/llcAsset_WBGroupLLC.json',
              er = Path.home()/ 'GDrive/Family/Assets/LLC-WBGroup/books/Accts/llcExpRev_WBGroupLLC.json')
    
def loadLedger(fn):
    jstr = fn.read_text()
    return json.loads(jstr)

def saveLedger(erList, fn):
    with open(fn,'w') as fio:
        json.dump(erList, fio)
    
def displayLedger(erList, **kwargs):
    for er in erList:
        amt = er['amt']
        pNm = er['propNm']
        aSub = er['acctSub']
        a = er['acct']
        l = er['Ledger']
    
        print(f"{pNm:13s} {amt:10.2f}: {er['aType']:7s}, {a} -> {l} ==acctSub:{aSub}")

def displayGroup(erList):
    df = pd.DataFrame(erList)
    return df.groupby(['acct', 'Ledger']).amt.sum()

def swapLedger(erList):
    for er in erList:
        amt = er['amt']
        pNm = er['propNm']
        aSub = er['acctSub']
        a = er['acct']
        l = er['Ledger']
        dc = er['aType']
        
        if er['acct'] != 'Acct.Cash.Bank' and er['Ledger'] == 'Acct.Cash.Bank':
            # swap ExpRev so Acct.Cash.banik
            er['acct'] = l
            er['Ledger'] = a
            er['aType'] = 'Credit' if dc == 'Debit' else 'Debit'
    
def cleanLedger(erOld):
    print(f"\n=========== Old ExpRev {len(erOld)} ==========")

    erList = []
    for er in erOld:
        # Clean transactions
        pNm = er['propNm']
        aSub = er['acctSub']
    
        a = er['acct']
        l = er['Ledger']
        dc = er['aType']
    

        # Ignore non RV_RV1
        if 'RV_RV1' != er['propNm']:                
            erList.append(er)
            continue
            
        # Ignore investment, purchase
        amt = er['amt']
        if amt == 177.00 :
            erList.append(er)
            #print(er)
            continue

        er['acct'] = 'Acct.Cash.Bank'
        er['Ledger'] = 'Acct.Fixed.Tangible.InConstruction'
        er['aType'] = 'Credit'
    
        print(f"{pNm:7s} {amt:10.2f}: {er['aType']:7s}, {a} -> {l} ==acctSub:{aSub}")
        erList.append(er)
    return erList

In [70]:
# diagonose llcExpRev: Clean up RV_RV1

# customize this lambda to filter only transactins desired
testRec = lambda er : er['propNm'] == 'RV_RV1'
testRec = lambda er : er['amt'] == 14.06
testRec = lambda er : 'Equity' in er['acct']

erList = loadLedger(fnDict['er'])

def displayLedger(erList, **kwargs):
    for er in erList:
        amt = er['amt']
        pNm = er['propNm']
        aSub = er['acctSub']
        a = er['acct']
        l = er['Ledger']

        func = kwargs.get('func', None)

        # If func and func is True - display, else ignore
        if func is not None:
            if not func(er) : continue
    
        print(f"{pNm:13s} {amt:10.2f}: {er['aType']:7s}, {a} -> {l} ==acctSub:{aSub} >>> tid:{er['tID']}")

#displayLedger(erList, func=testRec)
erList[0]

{'Ledger': 'Acct.Equity.Owner.Capital.Funds',
 '_unknown': '',
 'aType': 'Debit',
 'acct': 'Acct.Cash.Bank',
 'acctSub': 'Investment, Member FRojas',
 'amt': 219000.0,
 'desc': 'Owner investment',
 'dt': '2025.08.20',
 'propAddr': 'null',
 'propID': 'null',
 'propNm': 'H_805HighMesa',
 'propOwners': 'null',
 'refDB': 'llcBank',
 'refDoc': 'WT FED#02M03 NATIONAL FINANCIAL /ORG=FRANCIS X ROJAS SRF# 2780815232FS TRN#250820186189 RFB# 2780815232FS',
 'tDB': 'llcBank',
 'tID': '2025.08.20_D219000.00',
 'acctType': 'Equity'}

## Diagnose GL

- load GL
- proof: trace every GL transaction back to llc object


In [81]:
# stmtGL data

from ledger import setup_paths as _sp
from ledger.LLC import LLC
from ledger.stmtGL import stmtGL

_sp.load_config('WBGroupLLC', 2025)
llc = LLC('WBGroupLLC')
gl = stmtGL(llc)
glList = gl.load()
len(glList)

def testT(t):
    # return t['propNm'] == 'RV_RV1']

    a = t['acct']
    selA = 'Acct.Equity.Owner'
    return selA in a 
def testT2(t):
    # return t['propNm'] == 'RV_RV1']

    a = t['acct']
    selA = 'Acct.Fixed'
    return selA in a 

#pd.DataFrame([t for t in glList if abs(t['amt']) == 14.06])
df = pd.DataFrame([t for t in glList if testT2(t) ])
#df.acct.unique()
df.groupby(['refDB', 'propNm', 'propOwners','acctSub']).amt.agg(['sum','count'])

[setup_paths] Loaded 'WBGroupLLC/2025' from /Users/frankrojas/.llcRentalTracker/config.json → bus_repo=/Users/frankrojas/Library/CloudStorage/GoogleDrive-frankr6591@gmail.com/My Drive/Family/Assets/LLC-WBGroup


sum  \
refDB                    propNm        propOwners           acctSub                               
                         H_805HighMesa {"020250801_1": 100} YE:Acct.Exp.Depreciation    1903.13   
Assets/805HighMesa/Docs/ H_805HighMesa {"020250801_1": 100} Closing                   222322.89   
COA                                                                                        0.00   
llcBank                  RV_RV1                                                           54.08   
                                                            Const                        822.19   
                                                            Exp Other                    206.81   
                                       LLC:100%             Invest RV Rental             177.00   
llcPayable               H_805HighMesa o20250801_1:100%     Misc                         462.15   
                         RV_RV1        o20250801_1:100%     Const                        810.13   

                                                                                      count  
refDB                    propNm        propOwners           acctSub                          
                         H_805HighMesa {"020250801_1": 100} YE:Acct.Exp.Depreciation      1  
Assets/805HighMesa/Docs/ H_805HighMesa {"020250801_1": 100} Closing                       3  
COA                                                                                       6  
llcBank                  RV_RV1                                                           2  
                                                            Const                         9  
                                                            Exp Other                     5  
                                       LLC:100%             Invest RV Rental              1  
llcPayable               H_805HighMesa o20250801_1:100%     Misc                          1  
                         RV_RV1        o20250801_1:100%     Const                         3

In [86]:
## Diagnose Acct.Equity.Owner.Capital.Funds

def testEq(t):
    # return t['propNm'] == 'RV_RV1']

    a = t['acct']
    selA = 'Acct.Equity.Owner.Capital.Funds'
    return a == selA 

df = pd.DataFrame([t for t in glList if testEq(t) ])
df.groupby(['acct', 'propNm','propOwners']).amt.sum()

acct                             propNm         propOwners      
Acct.Equity.Owner.Capital.Funds                                          0.00
                                 H_805HighMesa                          30.00
                                                o20250801_1:100%    224912.15
                                                o20250801_1:96%        377.76
                                                o20250801_2:2%           7.87
                                                o20250801_3:2%           7.87
                                 RV_RV1         o20250801_1:100%       987.13
Name: amt, dtype: float64

In [59]:
import os
from pathlib import Path
sorted(os.listdir(Path('./docs/BUS')))

['.ipynb_checkpoints',
 'design_BUS_01-AccountingWorkflow.md',
 'design_BUS_01.1_AccountingDesign.md',
 'design_BUS_01.3-AccountingWorkflow.md',
 'design_BUS_01.5_BankIngestionAgent.md',
 'design_BUS_01.5_BusHealthAgent.md',
 'design_BUS_01.5_ClosingAid.md',
 'design_BUS_01.5_ExpRevAgent.md',
 'design_BUS_01.5_NewPropertyAgent.md',
 'design_BUS_01.5_TransactionEditModel.md',
 'design_BUS_01.9_YEClosing.md',
 'design_BUS_01.9_YEFinancialReport.md',
 'design_BUS_04.0_TaxPrep.md',
 'design_BUS_04.2_LLCTaxAgent.md',
 'design_BUS_04.6_Form1065Agent.md',
 'design_BUS_04.6_Form4562Agent.md',
 'design_BUS_04.6_Form8825Agent.md',
 'design_BUS_04.6_FormSchK1Agent.md',
 'design_BUS_04.8_BookToIRS_Aid.md',
 'design_BUS_04.8_IRS_Form1065_Notes.md',
 'design_BUS_04.8_IRS_Form4562_Notes.md',
 'design_BUS_04.8_IRS_SchK1_Notes.md',
 'design_BUS_04.8_Tax_BookToIRS.md']